# 02 — EDA: Weather Baselines by Mesoregion

**Objective**: Understand the climate baselines and anomaly patterns across Brazil's coffee-producing mesoregions.

**Data**: Production-weighted weather indices from Open-Meteo, aggregated to mesoregion level.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import DATA_PROCESSED_DIR, MESOREGIONS
from src.visualization.weather_maps import anomaly_heatmap, seasonal_climate_profile

weather = pd.read_parquet(DATA_PROCESSED_DIR / "weather_mesoregion_daily.parquet")
anomalies = pd.read_parquet(DATA_PROCESSED_DIR / "weather_anomalies.parquet")
print(f"Weather: {len(weather)} records, {weather.index.names}")
print(f"Anomalies: {len(anomalies)} records, {anomalies.index.names}")
weather.head()

## 2.1 Mesoregion Climate Profiles

Seasonal temperature and precipitation patterns for the top-producing mesoregions.

In [ ]:
top_mesos = ["31_10", "31_05", "31_12", "35_02", "32_04", "29_06"]
for mk in top_mesos:
    seasonal_climate_profile(mk).show()

## 2.2 Temperature Anomaly Heatmap

All 41 mesoregions × 78 months (2021–2026). Red = hotter than baseline, blue = cooler.

In [ ]:
anomaly_heatmap("temp_max_anom").show()

## 2.3 Precipitation Anomaly Heatmap

In [ ]:
anomaly_heatmap("precipitation_anom").show()

## 2.4 Anomaly Distributions

Histogram of temperature anomalies across all mesoregions — most days cluster near zero but extreme tails matter.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=anomalies["temp_max_anom"].dropna(),
    nbinsx=80,
    marker_color="#d4a373",
    opacity=0.7,
    name="Temp Max Anomaly",
))
fig.update_layout(
    title="Distribution of Daily Temperature Anomalies (All Mesoregions)",
    xaxis_title="Anomaly (°C)",
    yaxis_title="Frequency",
    height=400,
)
fig.show()

## 2.5 Precision Agriculture: Temperature by Altitude

High-altitude mesoregions (Sul de Minas, ~900–1,300m) vs. low-altitude (Cerrado, ~800–1,100m) show distinct temperature profiles.

In [ ]:
meso_data = anomalies.reset_index()
meso_data["meso_name"] = meso_data["meso_key"].map(
    lambda k: MESOREGIONS.get(k, {}).get("meso_name", k)
)

# Compare three climatically distinct mesoregions
compare = ["31_10", "31_05", "29_06"]
labels = [MESOREGIONS[k]["meso_name"] for k in compare]
subset = meso_data[meso_data["meso_key"].isin(compare)]

fig = go.Figure()
for i, mk in enumerate(compare):
    s = subset[subset["meso_key"] == mk]["temp_max_anom"].dropna()
    fig.add_trace(go.Box(y=s, name=labels[i], marker_color=["#4a3728", "#8b7355", "#c8b89a"][i]))

fig.update_layout(
    title="Temperature Anomaly Distribution by Mesoregion",
    yaxis_title="Temp Max Anomaly (°C)",
    height=450,
)
fig.show()

## Key Observations

- Fill in your observations here after running the cells above.